In [1]:
# =============================================================================
# BBO + PS-SW & PS-SWA Decision Tree Analysis (PS Metrics)
# Collects BBO history → PS-SW/PS-SWA trees → Scatter plot with R²
# =============================================================================

# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from mealpy.bio_based import BBO as MEALPY_BBO
from mealpy import IntegerVar
# YOUR PS-SW/PS-SWA IMPORTS (adjust paths)
from VarianceDecisionTree.PSDecisionTree import PSDecisionTree
from PSMiners.Mining import get_history_pRef  # or your BBO pRef function
import warnings
warnings.filterwarnings('ignore')

print("✅ PS-SW/PS-SWA + BBO analysis ready")


ModuleNotFoundError: No module named 'VarianceDecisionTree.PSDecisionTree'

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))  # Add current directory
sys.path.insert(0, os.path.abspath('..'))  # Add parent directory
sys.path.insert(0, os.path.abspath('PS-descriptors'))  # Add your project root

# Now your imports will work
import numpy as np
# ... rest of imports


In [ ]:
# Cell 2: BBO SETUP + RUN (Replace with your SAT problem)
dimension = 50  # Adjust to your problem
lb = tuple([0] * dimension)
ub = tuple([1] * dimension)  # Binary SAT
bounds = IntegerVar(lb=lb, ub=ub, name="bbo_sat")

# YOUR FITNESS FUNCTION
def fitness_func(solution):
    discrete_solution = np.array(solution, dtype=int)
    # REPLACE with: full_solution = FullSolution(discrete_solution)
    # return float(benchmark_problem.fitness_function(full_solution))
    fitness = np.sum(discrete_solution) * 0.5 + np.random.random() * 0.1  # DUMMY
    return float(fitness)

problem_dict = {"obj_func": fitness_func, "bounds": bounds, "minmax": "max", "save_population": True}

# Run BBO
model = MEALPY_BBO.OriginalBBO(epoch=100, pop_size=50, p_m=0.01, n_elites=2)
print("🚀 Running BBO...")
model.solve(problem_dict, mode='swarm')
print("✅ BBO completed")


In [ ]:
# Cell 3: EXTRACT BBO DATA + CREATE PRef
solutions = []
true_fitness = []

print("📊 Extracting BBO history...")
for population in model.history.list_population:
    for agent in population:
        x = np.array(agent.solution, dtype=int)
        f = float(agent.target.fitness)
        solutions.append(x)
        true_fitness.append(f)

X_bbo = np.array(solutions)
y_true = np.array(true_fitness)
print(f"✅ BBO data: {X_bbo.shape[0]} samples")


In [ ]:
# Cell 4: PS-SW TREE (simplicity variance)
print("🌳 Training PS-SW Decision Tree...")

ps_sw_tree = PSDecisionTree(
    max_depth=4,
    min_samples_leaf=20,
    metric="simplicity variance",  # PS-SW metric
    ps_budget=5000,
    ps_population=100,
    sample_size=10000
)

# Train PS-SW on BBO data
ps_sw_tree.fit(X_bbo, y_true)
y_pred_ps_sw = ps_sw_tree.predict(X_bbo)

r2_ps_sw = r2_score(y_true, y_pred_ps_sw)
print(f"✅ PS-SW R²: {r2_ps_sw:.4f}")


In [ ]:
# Cell 5: PS-SWA TREE (simplicity variance estimated_atomicity)
print("🌳 Training PS-SWA Decision Tree...")

ps_swa_tree = PSDecisionTree(
    max_depth=4,
    min_samples_leaf=20,
    metric="simplicity variance estimated_atomicity",  # PS-SWA metric
    ps_budget=5000,
    ps_population=100,
    sample_size=10000
)

ps_swa_tree.fit(X_bbo, y_true)
y_pred_ps_swa = ps_swa_tree.predict(X_bbo)

r2_ps_swa = r2_score(y_true, y_pred_ps_swa)
print(f"✅ PS-SWA R²: {r2_ps_swa:.4f}")


In [ ]:
# Cell 6: SAVE ALL DATA ARRAYS
np.save("bbo_y_true.npy", y_true)
np.save("bbo_y_pred_ps_sw.npy", y_pred_ps_sw)
np.save("bbo_y_pred_ps_swa.npy", y_pred_ps_swa)
np.save("bbo_X.npy", X_bbo)
print("💾 Saved: y_true, y_pred_ps_sw, y_pred_ps_swa, X_bbo")


In [ ]:
# Cell 7: DUAL SCATTER PLOT - PS-SW vs PS-SWA on BBO
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# PS-SW Plot
ax1.scatter(y_true, y_pred_ps_sw, alpha=0.6, s=30, c=y_true, cmap='viridis', edgecolor='k')
min_val = min(y_true.min(), y_pred_ps_sw.min())
max_val = max(y_true.max(), y_pred_ps_sw.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
ax1.set_xlabel('True Fitness (BBO)')
ax1.set_ylabel('PS-SW Predicted')
ax1.set_title(f'PS-SW on BBO\nR²={r2_ps_sw:.3f}')
ax1.grid(alpha=0.3)
ax1.legend()

# PS-SWA Plot
ax2.scatter(y_true, y_pred_ps_swa, alpha=0.6, s=30, c=y_true, cmap='plasma', edgecolor='k')
min_val = min(y_true.min(), y_pred_ps_swa.min())
max_val = max(y_true.max(), y_pred_ps_swa.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
ax2.set_xlabel('True Fitness (BBO)')
ax2.set_ylabel('PS-SWA Predicted')
ax2.set_title(f'PS-SWA on BBO\nR²={r2_ps_swa:.3f}')
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('bbo_ps_sw_vs_ps_swa.png', dpi=300, bbox_inches='tight')
plt.savefig('bbo_ps_sw_vs_ps_swa.pdf', bbox_inches='tight')
plt.show()

print("📈 Saved: bbo_ps_sw_vs_ps_swa.[png|pdf]")


In [ ]:
# Cell 8: COMPARISON TABLE FOR PAPER
comparison_data = {
    'Method': ['PS-SW', 'PS-SWA'],
    'Metric': ['simplicity variance', 'simplicity variance estimated_atomicity'],
    'R² Score': [r2_ps_sw, r2_ps_swa],
    'Tree Depth': [ps_sw_tree.tree_.max_depth, ps_swa_tree.tree_.max_depth],
    'Leaves': [ps_sw_tree.tree_.n_leaves, ps_swa_tree.tree_.n_leaves]
}

import pandas as pd
df = pd.DataFrame(comparison_data)
print("📊 PS-SW vs PS-SWA Comparison:")
print(df.round(4))

# Save table
df.to_latex('bbo_ps_comparison.tex', index=False)
df.to_csv('bbo_ps_comparison.csv')
print("💾 Saved LaTeX/CSV table for paper")


In [ ]:
# Cell 9: PS METRIC DETAILS (for paper discussion)
print("🔍 PS Tree Configuration Used:")
print("""
PS-SW:  "simplicity variance" 
   - Guides tree splits using partial solution simplicity + variance trade-off

PS-SWA: "simplicity variance estimated_atomicity" 
   - Enhanced metric incorporating atomicity for better explainability
   - Balances simplicity, variance reduction, AND atomic decision quality
""")

print(f"\nBBO Dataset: {len(y_true):,} samples")
print(f"Fitness range: [{y_true.min():.3f}, {y_true.max():.3f}]")
